# Practical 01: Building a Reproducible MLOps Workbench

**SCSE3040 Machine Learning Operations | Bennett University | Session 2026-27**

> This practical establishes the execution, dependency, randomness, data, and source-control habits that every later MLOps practical assumes.

| Item | Detail |
|---|---|
| Lectures | L01-L02 |
| Course Outcome | CO1 |
| Duration | 120 minutes |
| Memory | about 250 MB |
| GPU | not required |
| Marks | 10 |

## Aim

1. Verify which Python interpreter actually runs the notebook.
2. Record exact versions of important Python dependencies.
3. Control pseudo-random operations with explicit seeds.
4. Prove data identity with SHA-256.
5. Record run provenance and inspect Git provenance.

## Reproducibility in this practical

A virtual environment, pinned Python packages, controlled random states, data fingerprints, and source revision information substantially improve reproducibility.

They do not by themselves guarantee universal bit-for-bit equality across every operating system, processor, Python build, numerical library, or external runtime. The objective is to build disciplined evidence about the conditions that produced a result.


---

## Before you start

- Complete the repository setup instructions first.
- Start Jupyter using the course environment.
- Open this notebook from the P01 practical directory.
- Run cells in order from top to bottom.

### How to run a cell

Click a code cell and press **Shift + Enter**.

If notebook state becomes confusing, use **Kernel > Restart Kernel and Clear All Outputs**, then run again from the first cell.

A later cell may depend on variables created earlier. A `NameError` often means that an earlier required cell has not yet been executed.


In [ ]:
# Step 0: pre-flight verification.
# This cell inspects the environment. It does not install or modify packages.

import sys
from pathlib import Path

print("Python  :", sys.version.split()[0])
print("Program :", sys.executable)
print("Folder  :", Path.cwd())

_missing = []
for _name in ["numpy", "pandas", "sklearn"]:
    try:
        __import__(_name)
    except ImportError:
        _missing.append(_name)

for _name in ["numpy", "pandas", "sklearn"]:
    _mark = "missing" if _name in _missing else "ok"
    print(f"  {_name:<14} {_mark}")

if _missing:
    print()
    print("STOP. Missing imports:", ", ".join(_missing))
    print("First verify that Jupyter is using the intended course Python.")
else:
    print()
    print("All good. Continue to Step 1.")


---

## Step 0b: ensure the shared dataset exists

Every practical uses the same 600 synthetic food-delivery observations.

If the shared CSV is not available at the expected relative path, the next cell rebuilds it deterministically from the course seed.


In [ ]:
# Shared delivery-time dataset.
# The generator is deterministic for the specified NumPy generator, seed,
# and software context used by this practical.

import csv
from pathlib import Path

import numpy as np

SEED = 42
N_ROWS = 600
DATA = Path("..") / "data" / "delivery_times.csv"


def make_delivery_csv(path=DATA, seed=SEED):
    """Write the 600-row synthetic delivery dataset."""
    rng = np.random.default_rng(seed)

    distance_km = np.round(rng.uniform(0.5, 12.0, N_ROWS), 2)
    prep_time_min = np.round(rng.uniform(5, 30, N_ROWS), 0)
    traffic_level = rng.integers(1, 4, N_ROWS)
    rain = rng.binomial(1, 0.25, N_ROWS)

    delivery_min = np.round(
        6.0
        + 3.1 * distance_km
        + 0.65 * prep_time_min
        + 4.2 * traffic_level
        + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS),
        1,
    )

    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", newline="", encoding="utf-8") as fh:
        writer = csv.writer(fh)
        writer.writerow([
            "distance_km",
            "prep_time_min",
            "traffic_level",
            "rain",
            "delivery_min",
        ])

        for i in range(N_ROWS):
            writer.writerow([
                distance_km[i],
                int(prep_time_min[i]),
                int(traffic_level[i]),
                int(rain[i]),
                delivery_min[i],
            ])

    return path


if not DATA.exists():
    make_delivery_csv()
    print("dataset rebuilt ->", DATA)
else:
    print("dataset found   ->", DATA)


---

# Walkthrough

Read each step, then run its cell.

### Step 1: Which Python is actually executing this notebook?

A machine may contain several Python installations. The notebook kernel determines which interpreter evaluates each cell.

The strongest local check is `sys.executable`, because it reports the executable of the running Python process.


In [ ]:
import sys
from pathlib import Path

print("Python version :", sys.version.split()[0])
print("Python program :", sys.executable)

normalized = sys.executable.replace("\\", "/")
in_venv = "/.venv/" in normalized or normalized.endswith("/.venv/Scripts/python.exe")

print("Inside .venv   :", in_venv)
print("Working folder :", Path.cwd())


If `Inside .venv` is `False`, stop before debugging later ML code.

First verify the notebook kernel and compare `sys.executable` with the project-local Python interpreter expected by the course setup.


### Step 2 --- What is installed in this environment?

A library is code somebody else wrote that you can use. Your project
depends on several. Each has a **version number**, and versions
matter: `scikit-learn` 1.9 does not behave exactly like 1.2.

Let us ask Python what it has.

In [ ]:
from importlib.metadata import version

LIBRARIES = ["numpy", "pandas", "scikit-learn", "matplotlib"]

for name in LIBRARIES:
    print(f"{name:<15} {version(name)}")

Record these versions as part of the execution context.

Version information is important because a package name alone does not identify the release whose behavior produced an experiment.


### Step 3: Write pinned versions to `requirements.txt`

A pinned requirement uses:

```text
name==version
```

The `==` requests one exact package version.

This practical writes a small learning artifact under `work/`. The repository-level course lock file remains the authoritative full environment specification.


In [ ]:
WORK = Path("work")
WORK.mkdir(exist_ok=True)

lines = [f"{name}=={version(name)}" for name in LIBRARIES]
(WORK / "requirements.txt").write_text("\n".join(lines) + "\n",
                                       encoding="utf-8")

print("wrote", WORK / "requirements.txt")
print("-" * 40)
print((WORK / "requirements.txt").read_text(encoding="utf-8"))

A pinned requirements file can be installed with the interpreter-explicit command:

```powershell
python -m pip install -r requirements.txt
```

Using `python -m pip` reduces ambiguity when several Python installations exist.

A requirements file records Python package dependencies. It does not capture every operating-system, hardware, environment-variable, or external-service dependency of a production system.


### Step 4 --- Random numbers change every time you ask

Now the third idea: randomness. Run the next cell, then run it a
second time. Look at the numbers.

In [ ]:
import numpy as np

careless = np.random.default_rng()   # no seed given
print("three random numbers:", np.round(careless.uniform(0, 10, 3), 2))

Run the previous cell more than once.

Different values are expected when no explicit seed is supplied. Variability is not a failure. The engineering question is whether the experiment requires repeatability.


### Step 5: A seed makes pseudo-random operations repeatable

A **seed** selects a reproducible initial state for a pseudo-random number generator.

For the same generator implementation and compatible software context, using the same seed allows the same pseudo-random sequence to be reproduced.

The seed does not remove randomness, and it does not guarantee identical behavior for every library, algorithm, platform, or future implementation.


In [ ]:
first  = np.random.default_rng(42).uniform(0, 10, 3)
second = np.random.default_rng(42).uniform(0, 10, 3)

print("first run :", np.round(first, 2))
print("second run:", np.round(second, 2))
print("identical :", np.array_equal(first, second))

The durable habit is:

> Identify every stochastic component that matters and set or record its random state when repeatability is required.

Later practicals will apply this principle to data splitting and randomized models.


### Step 6: Build the delivery dataset twice

Now use deterministic generation for a real course artifact.

The notebook writes the synthetic delivery dataset twice and compares full SHA-256 digests.

Matching digests provide extremely strong evidence that the generated files contain identical bytes.


In [ ]:
import hashlib

def sha256_of(path):
    """A short fingerprint of a file's exact contents."""
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

make_delivery_csv(WORK / "run_a.csv")
make_delivery_csv(WORK / "run_b.csv")

fp_a = sha256_of(WORK / "run_a.csv")
fp_b = sha256_of(WORK / "run_b.csv")

print("run A:", fp_a[:16], "...")
print("run B:", fp_b[:16], "...")
print("identical files:", fp_a == fp_b)

Two independent generation calls produced matching byte content.

This demonstrates a stronger reproducibility check than visual comparison.

A SHA-256 digest is a content-derived identifier. It does not contain the original dataset and cannot reconstruct it.


### Step 7 --- Look at the data you just made

**CSV** (Comma Separated Values) is a plain text table. `pandas` is
the library that reads one into something you can work with.

In [ ]:
import pandas as pd

orders = pd.read_csv(DATA)

print("rows, columns:", orders.shape)
print()
print(orders.head())
print()
print(orders.describe().round(1))

`distance_km` is how far the rider goes, `prep_time_min` is how long
the restaurant takes, `traffic_level` is 1 to 3, `rain` is 0 or 1,
and `delivery_min` is the answer we will eventually try to predict.

### Step 8: Record run provenance

A model score is weak evidence if we cannot later determine the execution conditions that produced it.

The next cell writes a compact JSON experiment manifest with Python, platform, seed, row count, data fingerprint, package versions, and Git commit information when available.


In [ ]:
import json
import platform
import subprocess


def current_git_commit():
    """Return the current repository commit when Git provenance is available."""
    try:
        result = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=Path.cwd(),
            capture_output=True,
            text=True,
            check=True,
        )
        return result.stdout.strip()
    except Exception:
        return None


run_info = {
    "python": sys.version.split()[0],
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "seed": SEED,
    "rows": len(orders),
    "data_sha256": sha256_of(DATA),
    "git_commit": current_git_commit(),
    "libraries": {name: version(name) for name in LIBRARIES},
}

(WORK / "run_info.json").write_text(
    json.dumps(run_info, indent=2),
    encoding="utf-8",
)

print(json.dumps(run_info, indent=2))


### Step 9: Inspect Git provenance

The public course repository is already a Git repository, so this practical does **not** create a nested `.git` directory inside `work/`.

A commit identifies one recorded source-code state.

The next cell only inspects repository information. It does not create a commit.


In [ ]:
import subprocess


def run_git(*args):
    """Run a read-only Git command and return its output."""
    done = subprocess.run(
        ["git", *args],
        cwd=Path.cwd(),
        capture_output=True,
        text=True,
    )

    print("$ git", " ".join(args))
    output = (done.stdout + done.stderr).strip()
    print(output or "(no output)")
    print("-" * 60)
    return done


run_git("status", "--short")
run_git("rev-parse", "--show-toplevel")
run_git("log", "-1", "--oneline")


You now have the core evidence P01 is meant to establish:

- execution identity;
- dependency identity;
- controlled pseudo-randomness;
- data identity;
- a run manifest;
- source-code provenance.

Later tools such as MLflow, containers, CI, and monitoring will automate or extend these controls rather than replace the underlying discipline.


---

# Your turn

The walkthrough above is finished. Now you write some code.

There are **3 tasks**. Each one is small. Each one has a hint.
Do them in order.

Where you see `# TODO`, replace that line with your own code. Do not delete
the variable name on the left of the `=` sign --- the self-check at the
bottom looks for exactly that name.

When you have tried all three, run the **self-check** cell at the end. It
prints a table telling you which tasks are correct. You can run it as many
times as you like.

### Task T1: Change the seed

The shared dataset uses seed `42`.

Using seed `7`:

1. create a NumPy generator;
2. draw 600 values uniformly between `0.5` and `12.0`;
3. round to two decimal places;
4. take the first three values;
5. store them as a plain Python list named `T1_first_three`.

#### Progressive hint

Start with:

```python
rng = np.random.default_rng(7)
```

Then use `.uniform(...)`, `np.round(...)`, slicing, and `list(...)`.

Do not copy a precomputed answer.


In [ ]:
# TODO: replace None with your one-line answer.
T1_first_three = None

print("T1_first_three =", T1_first_three)

### Task T2: Write your own pinned requirements file

Create:

```text
work/my_requirements.txt
```

with exactly three non-empty lines for:

- `numpy`
- `pandas`
- `scikit-learn`

Each line must use the version installed in the current interpreter:

```text
name==version
```

Do not type version numbers manually.

#### Hint

You already used `version(name)` in Step 2. Build one formatted line per package, join them with newline characters, and write the result with `Path.write_text(...)`.


In [ ]:
MY_LIBS = ["numpy", "pandas", "scikit-learn"]

# TODO: build one "name==version" line per library, then write them
#       to work/my_requirements.txt with a newline between them.
my_lines = None

print(my_lines)

### Task T3: Fingerprint a run

Implement:

```python
fingerprint(path)
```

so that it returns a dictionary with exactly these keys:

- `"rows"`: number of CSV data rows, excluding the header;
- `"sha256"`: the complete SHA-256 digest of the file;
- `"seed"`: the practical's `SEED`.

Then call it on `DATA` and store the result in `T3_fp`.

#### Hint

`pd.read_csv(path)` gives a DataFrame, `len(...)` gives the number of data rows, and the helper `sha256_of(path)` is already available.


In [ ]:
def fingerprint(path):
    # TODO: return a dictionary with the keys rows, sha256 and seed.
    return None


T3_fp = fingerprint(DATA)
print(T3_fp)

---

## Self-check

Run the cell below to mark your work.

In [ ]:
# ------------------------------------------------------------------
# SELF-CHECK
# ------------------------------------------------------------------

_results = []


def _check(label, fn):
    """Evaluate one condition without breaking the notebook."""
    try:
        ok = bool(fn())
    except Exception:
        ok = False
    _results.append((label, ok))


_check(
    "T1 | T1_first_three holds three numbers",
    lambda: len(T1_first_three) == 3,
)

_check(
    "T1 | values match the seed-7 generation procedure",
    lambda: all(
        abs(float(a) - float(b)) < 1e-9
        for a, b in zip(
            T1_first_three,
            np.round(
                np.random.default_rng(7).uniform(0.5, 12.0, 600),
                2,
            )[:3],
        )
    ),
)

_check(
    "T2 | work/my_requirements.txt exists",
    lambda: (WORK / "my_requirements.txt").is_file(),
)


def _t2_versions_are_exact():
    path = WORK / "my_requirements.txt"
    lines = [
        line.strip()
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    expected = {
        name: version(name)
        for name in ["numpy", "pandas", "scikit-learn"]
    }

    if len(lines) != 3:
        return False

    parsed = {}

    for line in lines:
        if line.count("==") != 1:
            return False
        name, ver = [part.strip() for part in line.split("==")]
        parsed[name] = ver

    return parsed == expected


_check(
    "T2 | all three packages are pinned to installed versions",
    _t2_versions_are_exact,
)

_check(
    "T3 | fingerprint() returns exactly the required keys",
    lambda: set(T3_fp) == {"rows", "sha256", "seed"},
)

_check(
    "T3 | it counts 600 rows and records seed 42",
    lambda: T3_fp["rows"] == 600 and T3_fp["seed"] == 42,
)

_check(
    "T3 | checksum matches the file and is a full SHA-256 digest",
    lambda: (
        T3_fp["sha256"] == sha256_of(DATA)
        and len(T3_fp["sha256"]) == 64
    ),
)

print("=" * 66)
print("SELF-CHECK   Practical 01: Reproducible MLOps Workbench")
print("=" * 66)

for _label, _ok in _results:
    print(f"  [{'PASS' if _ok else 'FAIL'}]  {_label}")

print("-" * 66)

_passed = sum(1 for _, _ok in _results if _ok)
print(f"  {_passed} of {len(_results)} checks passed")
print("=" * 66)

if _passed == len(_results):
    print("Well done. Save the notebook using the required P01 filename.")
else:
    print("Read the FAIL lines, fix only those tasks, then run this cell again.")


---

## Submission

Follow the LMS instructions given by your instructor.

A typical submission contains:

1. this notebook, executed top to bottom with outputs visible;
2. `work/my_requirements.txt`;
3. any additional evidence explicitly requested by the instructor.

Rename the notebook:

```text
P01_<your-roll-number>.ipynb
```

### Marking

| Component | Marks |
|---|---:|
| Walkthrough completed with outputs visible | 3 |
| T1: controlled randomness | 2 |
| T2: correctly pinned requirements | 2 |
| T3: working fingerprint function | 3 |
| **Total** | **10** |

## Read more

- Python virtual environments: <https://docs.python.org/3/tutorial/venv.html>
- pip requirements files: <https://pip.pypa.io/en/stable/reference/requirements-file-format/>
- NumPy random generator: <https://numpy.org/doc/stable/reference/random/generator.html>
- Python `hashlib`: <https://docs.python.org/3/library/hashlib.html>
- Pro Git: <https://git-scm.com/book/en/v2>
